In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.output.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .getOrCreate()

df = spark.read.format("com.mongodb.spark.sql.DefaultSource").load()

In [2]:
df.printSchema()
df.show(5)

root
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+-------------------+----------------+--------------------+-----+
|                 _id|      fecha_captura|           grupo|       identificador|valor|
+--------------------+-------------------+----------------+--------------------+-----+
|{69ddaba86a1fc7ee...|2026-04-14 02:51:16|Nombre del grupo|A Light in the Attic|51.77|
|{69ddaba86a1fc7ee...|2026-04-14 02:51:16|Nombre del grupo|  Tipping the Velvet|53.74|
|{69ddaba86a1fc7ee...|2026-04-14 02:51:16|Nombre del grupo|          Soumission| 50.1|
|{69ddaba86a1fc7ee...|2026-04-14 02:51:16|Nombre del grupo|       Sharp Objects|47.82|
|{69ddaba86a1fc7ee...|2026-04-14 02:51:16|Nombre del grupo|Sapiens: A Brief ...|54.23|
+--------------------+-------------------+----------------+

In [3]:
from pyspark.sql import functions as F

df.groupBy("grupo").agg(
    F.count("*").alias("cantidad"),
    F.avg("valor").alias("promedio"),
    F.max("valor").alias("maximo"),
    F.min("valor").alias("minimo")
).show()

+----------------+--------+-----------------+------+------+
|           grupo|cantidad|         promedio|maximo|minimo|
+----------------+--------+-----------------+------+------+
|Nombre del grupo|      60|35.00266666666666| 57.31| 12.84|
+----------------+--------+-----------------+------+------+



In [4]:
df.orderBy(F.desc("valor")).show(1)


+--------------------+-------------------+----------------+--------------------+-----+
|                 _id|      fecha_captura|           grupo|       identificador|valor|
+--------------------+-------------------+----------------+--------------------+-----+
|{69ddaba96a1fc7ee...|2026-04-14 02:51:19|Nombre del grupo|Slow States of Co...|57.31|
+--------------------+-------------------+----------------+--------------------+-----+
only showing top 1 row



In [5]:
df.orderBy("valor").show(1)

+--------------------+-------------------+----------------+-------------+-----+
|                 _id|      fecha_captura|           grupo|identificador|valor|
+--------------------+-------------------+----------------+-------------+-----+
|{69ddaba96a1fc7ee...|2026-04-14 02:51:17|Nombre del grupo|  In Her Wake|12.84|
+--------------------+-------------------+----------------+-------------+-----+
only showing top 1 row



In [ ]:
df.filter(df.valor > 50).show()